In [ ]:
#------------------------------------------------------ Part 1 - Dataset Creation (Without Data Visualization) ------------------------------------------------------#

# not complete code from part 1, just the necessary code
import pandas as pd
import matplotlib.pyplot as plt # data visualization
import seaborn as sns # data visualization
from datetime import date # to calculate ages of users
from IPython.display import display # to display decriptive statistics
from google.colab import files # to upload files
import numpy as np
import re # for text analysis
import pytz # for timezone conversions
from zoneinfo import ZoneInfo # for timezone conversions
from sklearn.preprocessing import MinMaxScaler





'''for text analysis'''
import string
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.util import ngrams

nltk.download('stopwords')


# creating an auxilliary df that stores all of the columns as strings for easier filtering
# removing "?" from the target variable column
df_str = df.astype(str)
df_str = df_str[df_str['sentiment'] != '?']


"""$\textbf{פונקציות לעיבוד נתונים גולמיים - יצירת פיצ'רים}$"""

'''REGION'''
# mapping of timezone abbreviations to regions or continents - for informative purposes
timezone_to_region = {
    # timezones found in data using unique() method
    'AEST': 'Australia',
    'EET': 'Europe/Eastern Europe',
    'GMT': 'Europe/Africa',
    'MSK': 'Europe/Russia',
    'PST': 'North America/United States',
    'CET': 'Europe/Central Europe',
    'EST': 'North America/United States',
    'KST': 'Asia/South Korea',
    'UTC': 'Global',
    'JST': 'Asia/Japan'
}


'''HOUR OF POST'''
# mapping of timezone abbreviations to regions or continents - for time conversion purposes
timezone_to_iana = {
    'AEST': 'Australia/Sydney',
    'EET': 'Europe/Helsinki',
    'GMT': 'Etc/GMT',
    'MSK': 'Europe/Moscow',
    'PST': 'America/Los_Angeles',
    'CET': 'Europe/Paris',
    'EST': 'America/New_York',
    'KST': 'Asia/Seoul',
    'UTC': 'UTC',
    'JST': 'Asia/Tokyo'
}


def calculate_age(birth_date, post_datetime):
    """Calculates age at the time of the post."""
    birth_date = pd.to_datetime(birth_date)
    post_date = pd.to_datetime(post_datetime)

    return post_date.year - birth_date.year - (
        (post_date.month, post_date.day) < (birth_date.month, birth_date.day)
    )


def localize_post_time(post_time, timezone_key):
    """
    Localizes a naive datetime to the correct timezone using IANA mapping.
    If already timezone-aware, converts to the correct timezone.
    Uses modern zoneinfo (Python 3.9+).

    Args:
        post_time (datetime): naive or aware datetime
        timezone_key (str): abbreviation like 'EST', 'CET', etc.

    Returns:
        datetime: timezone-aware datetime
    """
    iana_name = timezone_to_iana.get(timezone_key)
    if not iana_name:
        raise ValueError(f"Unknown timezone key: {timezone_key}")

    if post_time.tzinfo is None:
        # Naive: attach timezone directly
        return post_time.replace(tzinfo=ZoneInfo(iana_name))
    else:
        # Aware: convert to target timezone
        return post_time.astimezone(ZoneInfo(iana_name))


def get_slot_of_day(dt):
  """
  Extracts the hour from a datetime object and determines if it's morning, noon,
  afternoon, evening, or night.

  Args:
    dt: A datetime object.

  Returns:
    A string indicating the time of day ('Morning', 'Noon', 'Afternoon', 'Evening', or 'Night').
  """
  hour = dt.hour
  if 6 <= hour < 12:
    return 'Morning'
  elif 12 <= hour < 14:
    return 'Noon'
  elif 14 <= hour < 18:
    return 'Afternoon'
  elif 18 <= hour <= 23:
    return 'Evening'
  else:
    return 'Night'



def classify_and_get_content_type(url):
    """
    Determines the content type of a URL based on its extension or domain.

    Args:
    url: A string containing a URL.

    Returns:
    A string representing the type of content:
      - 'missing' if the URL is NaN or an empty string.
      - 'image' if the URL ends with an image extension like .jpg, .jpeg, .png, .gif, .bmp, or webp.
      - 'video' if the URL belongs to a video platform like YouTube, Vimeo, or Dailymotion.
      - 'other' for URLs with an unsupported extension.
      - 'link' for any other URL type not matching the above categories.
    """
    # Check if the URL is missing or empty
    if pd.isna(url) or str(url).strip() == '':
        return 'missing'

    # Convert the URL to lowercase and strip any surrounding whitespace
    url = url.lower().strip()

    # Split the URL and get the last part (after the last '/')
    last_part = url.split('/')[-1]

    # If the URL contains a file extension
    if '.' in last_part:
        # Extract the file extension
        ext = last_part.split('.')[-1].split('?')[0].split('#')[0]

        # Check the file extension and categorize it
        if ext in ['jpg', 'jpeg', 'png', 'gif', 'bmp', 'webp']:
            return 'image'
        elif ext in ['mp4', 'avi', 'mov', 'wmv', 'mkv', 'webm']:
            return 'video'
        else:
            return 'other'

    # If the URL belongs to a known video platform
    elif re.search(r'youtube\.com|vimeo\.com|dailymotion\.com', url, re.IGNORECASE):
        return 'video'

    # If the URL ends with .pdf
    elif re.search(r'\.pdf$', url, re.IGNORECASE):
        return 'pdf'

    # If the URL ends with an audio extension
    elif re.search(r'\.mp3|\.wav|\.flac', url, re.IGNORECASE):
        return 'audio'

    # If none of the above conditions match, it's a regular link
    return 'link'




def username_in_email_soft(username, email):
    if pd.isna(username) or pd.isna(email):
        return False

    username_clean = ''.join(re.findall(r'[a-zA-Z]+', str(username))).lower()
    email = str(email).lower()

    parts = [username_clean[i:i+4] for i in range(len(username_clean)-2)]
    return any(part in email for part in parts) if parts else False


"""### $\textbf{עיבוד נתונים ויצירת דאטהפריים חדש עם משתנים נוספים}$"""


'''PRE-PROCESSING'''

df_analysis = df.copy()

df_analysis = df_analysis.astype('string')

'''PRE-PROCESSING - CHECKING FOR QUESTION MARKS (MISSING VALUES)'''

rows_with_question_marks = df.apply(lambda row: row.str.contains('\?').any(), axis=1).sum()

print(f"Number of rows with at least one question mark: {rows_with_question_marks}")

df_clean_target = df[df['sentiment'] != "?"]
rows_with_question_marks_excluding_target = df_clean_target.apply(lambda row: row.str.contains('\?').any(), axis=1).sum()
print(f"Number of rows with at least one question mark excluding the missing values in the target variable: {rows_with_question_marks_excluding_target}")

'''PRE-PROCESSING - CHECKING FOR DUPLICATES'''
df_analysis.duplicated().sum()     # count duplicates

'''REMOVING OUTLIERS USING THE IQR METHOD'''

def remove_outliers_iqr(df):
    """
    Removes outliers from all numerical columns in the DataFrame using the IQR method.

    Parameters:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A new DataFrame with outliers removed.
    """
    df_clean = df.copy()
    numeric_cols = df_clean.select_dtypes(include='number').columns

    for col in numeric_cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        mask = (df_clean[col] >= Q1 - 1.5 * IQR) & (df_clean[col] <= Q3 + 1.5 * IQR)
        df_clean = df_clean[mask]

    return df_clean

'''FILTERED DATAFRAME'''

df_analysis = remove_outliers_iqr(df_analysis)

'''PRE-PROCESSING - CHECKING FOR QUESTION MARKS (MISSING VALUES) AFTER REMOVING OUTLIERS'''

rows_with_question_marks_outliers_removed = df_analysis.apply(lambda row: row.str.contains('\?').any(), axis=1).sum()

print(f"Number of rows with at least one question mark after removing outliers: {rows_with_question_marks_outliers_removed}")

'''PRE-PROCESSING - REMOVING SAMPLES WHERE THERE ARE AT LEAST THAN 8 MISSING VALUES'''

# Count number of '?' per row
missing_counts = (df_analysis == "?").sum(axis=1)

# Keep only rows with fewer than 8 question marks
df_analysis = df_analysis[missing_counts < 8]

'''PRE-PROCESSING - CHECKING FOR QUESTION MARKS (MISSING VALUES) AFTER REMOVING SAMPLES WITH AT LEAST 8 MISSING VALUES'''

rows_with_question_marks_extreme = df_analysis.apply(lambda row: row.str.contains('\?').any(), axis=1).sum()

print(f"Number of rows with at least one question mark after handling extreme number of missing values: {rows_with_question_marks_extreme}")

'''PRE-PROCESSING - HANDLING MISSING VALUES PER COLUMN'''
df_analysis = df_analysis[df_analysis['sentiment'] != "?"]

def fill_categorical_missing_by_probability_columns(df, columns):
    if isinstance(columns, str):
        columns = [columns]  # if a single column is passed as a string, make it a list

    for column in columns:
        # Replace "?" with np.nan first
        df[column] = df[column].astype(str)
        df[column] = df[column].replace("?", np.nan)

        # Get non-missing values
        non_missing = df[column].dropna()

        if non_missing.empty:
            continue  # skip if no non-missing values to sample from

        # Calculate probabilities for each category
        value_counts = non_missing.value_counts(normalize=True)

        # Find missing indexes (rows) that need to be filled
        missing_idx = df[df[column].isna()].index

        if len(missing_idx) == 0:
            continue  # skip if there are no missing values

        # Randomly sample based on probabilities
        sampled_values = np.random.choice(value_counts.index, size=len(missing_idx), p=value_counts.values)

        # Fill missing
        df.loc[missing_idx, column] = sampled_values

    return df

def fill_numerical_missing_columns(df, columns):
    if isinstance(columns, str):
        columns = [columns]  # if a single column is passed as a string, make it a list

    for column in columns:
        # Convert column to string to identify '?'
        df[column] = df[column].astype('string')
        df[column] = df[column].replace("?", np.nan)
        non_missing = df[column].dropna()

        non_missing_values = pd.to_numeric(df[column], errors='coerce')  # convert to float
        df[column] = pd.to_numeric(df[column], errors='coerce')  # convert to float

        median_value = non_missing_values.median()

        # Replace '?' with the median value
        df[column] = df[column].fillna(median_value)

        # Convert the column back to numeric after replacing '?' with the median
        df[column] = pd.to_numeric(df[column], errors='coerce')

    return df

df_analysis = fill_numerical_missing_columns(df_analysis, ['post_likes','posts_retweets','followers','previous_posts_count'])

df_analysis = fill_categorical_missing_by_probability_columns(df_analysis, ['type', 'checkmark', 'timezone'])

# converting numerical columns to numeric data type
df_analysis['post_likes'] = pd.to_numeric(df_analysis['post_likes'], errors='coerce')
df_analysis['posts_retweets'] = pd.to_numeric(df_analysis['posts_retweets'], errors='coerce')
df_analysis['followers'] = pd.to_numeric(df_analysis['followers'], errors='coerce')
df_analysis['previous_posts_count'] = pd.to_numeric(df_analysis['previous_posts_count'], errors='coerce')

# setting datetime-type columns
df_analysis['post_datetime'] = pd.to_datetime(df_analysis['post_datetime'], errors='coerce')
df_analysis['account_creation_date'] = pd.to_datetime(df_analysis['account_creation_date'], errors='coerce')
df_analysis['birthday'] = pd.to_datetime(df_analysis['birthday'], errors='coerce')

'''CHECKING FOR QUESTION MARKS (MISSING VALUES) AFTER FILLING MISSING VALUES'''

rows_with_question_marks_assigned_values = df_analysis.apply(lambda row: row.str.contains('\?').any(), axis=1).sum()

print(f"Number of rows with at least one question mark after assigning values: {rows_with_question_marks_assigned_values}")

'''PRE-PROCESSING - CONVERTING DATA TYPES AND FORMATTING'''

'''MONTH OF POST'''
df_analysis['month_of_post'] = df_analysis['post_datetime'].dt.month_name() # January, February...

'''LOCALIZED POST DATETIME'''
df_analysis['localized_post_datetime'] = [
    localize_post_time(post_time, timezone)
    for post_time, timezone in zip(df_analysis['post_datetime'], df_analysis['timezone'])
]



'''MANIPULATING VARIABLES AFTER PRE-PROCESSING'''
'''ASSUMPTION: ALL OF THE DATA TYPES ARE COORDINATED AND ALIGNED'''

'''REGION'''
df_analysis['region'] = df_analysis['timezone'].map(timezone_to_region)
df_analysis['region'] = df_analysis['region'].astype('string')


'''DAY OF WEEK'''
df_analysis['day_of_week'] = df_analysis['post_datetime'].dt.day_name()
df_analysis['day_of_week'] = df_analysis['day_of_week'].astype('string')


'''AGE'''
# transform birthday to age at post using apply
df_analysis['age'] = df_analysis.apply(lambda row: calculate_age(row['birthday'], row['localized_post_datetime']), axis=1)
df_analysis['age'] = pd.to_numeric(df_analysis['age'])

'''LOCALIZED SLOT OF DAY'''
df_analysis['localized_slot_of_day'] = df_analysis['localized_post_datetime'].apply(get_slot_of_day)
df_analysis['localized_slot_of_day'] = df_analysis['localized_slot_of_day'].astype('string')


'''DAILY POST RETWEET RATE'''
df_analysis['daily_retweet_rate'] = df_analysis['posts_retweets'] / 30
df_analysis['daily_retweet_rate'] = pd.to_numeric(df_analysis['daily_retweet_rate'])



'''ACCOUNT SENIORITY (AGE AT POST) IN WEEKS'''
# calculating account seniority in weeks according to post date
df_analysis['account_seniority_days'] = (df_analysis['post_datetime'] - df_analysis['account_creation_date']).dt.days
df_analysis['account_seniority_days'] = pd.to_numeric(df_analysis['account_seniority_days'])

df_analysis['account_seniority_weeks'] = df_analysis['account_seniority_days'] / 7
# replacing 0 weeks with 1 to avoid division by zero
df_analysis['account_seniority_weeks'] = df_analysis['account_seniority_weeks'].replace(0, 1)
df_analysis['account_seniority_weeks'] = pd.to_numeric(df_analysis['account_seniority_weeks'])


'''POSTS PER WEEK'''
# calculating posts per week
df_analysis['posts_per_week'] = df_analysis['previous_posts_count'] / df_analysis['account_seniority_weeks']
df_analysis['posts_per_week'] = pd.to_numeric(df_analysis['posts_per_week'])



'''FOLLOWERS PER WEEK'''
df_analysis['followers_per_week'] = df_analysis['followers'] / df_analysis['account_seniority_weeks']
df_analysis['followers_per_week'] = pd.to_numeric(df_analysis['followers_per_week'])


'''FOLLOWERS TO PREVIOUS POSTS RATE'''
df_analysis['followers_to_previous_posts'] = df_analysis['followers'] / df_analysis['previous_posts_count']
df_analysis['followers_to_previous_posts'] = pd.to_numeric(df_analysis['followers_to_previous_posts'])


'''LIKES TO RETWEETS RATE'''
df_analysis['likes_to_retweets_rate'] = (df_analysis['post_likes'] / df_analysis['posts_retweets'])
df_analysis['likes_to_retweets_rate'] = pd.to_numeric(df_analysis['likes_to_retweets_rate'])


'''LIKES TO FOLLOWERS RATE'''
df_analysis['likes_to_followers_rate'] = df_analysis['post_likes'] / df_analysis['followers']
df_analysis['likes_to_followers_rate'] = pd.to_numeric(df_analysis['likes_to_followers_rate'])


'''TEXT WORD COUNT'''
df_analysis['text_word_count'] = df_analysis['text'].apply(lambda x: len(x.split()))
df_analysis['text_word_count'] = pd.to_numeric(df_analysis['text_word_count'])


'''TEXT # COUNT'''
df_analysis['hashtag_count'] = df_analysis['text'].apply(lambda x: len([word for word in x.split() if word.startswith('#')]))
df_analysis['hashtag_count'] = pd.to_numeric(df_analysis['hashtag_count'])


'''ATTACHED CONTENT TYPE IN POST'''
df_analysis['attached_content_type'] = df_analysis['embedded_content_url'].apply(classify_and_get_content_type)
df_analysis['attached_content_type'] = df_analysis['attached_content_type'].astype('string')


'''USER HAS PROFILE PICTURE'''
df_analysis['user_has_profile_picture'] = df_analysis['profile_picture'].apply(
    lambda x: 0 if pd.isna(x) or x == "?" else 1
)
df_analysis['user_has_profile_picture'] = pd.to_numeric(df_analysis['user_has_profile_picture'])



'''COMBINED FEATURE FOR ATTACHED CONTENT TYPE AND PICTURE'''
df_analysis['attached_content_and_picture_status'] = df_analysis.apply(
    lambda row: f"{row['attached_content_type']}_with_pic" if row['user_has_profile_picture'] == 1 else f"{row['attached_content_type']}_no_pic",
    axis=1
)
df_analysis['attached_content_and_picture_status'] = df_analysis['attached_content_and_picture_status'].astype('string')


'''COMBINED FEATURE FOR CHECKMARK (TYPE OF ACCOUNT) AND PROFILE PICTURE'''
df_analysis['profile_picture_and_checkmark_status'] = df_analysis.apply(
    lambda row: f"{row['user_has_profile_picture']}_{row['checkmark']}", axis=1)
df_analysis['profile_picture_and_checkmark_status'] = df_analysis['profile_picture_and_checkmark_status'].astype('string')


'''COMBINED FEATURE FOR PROFILE PICTURE AND USERNAME IN EMAIL'''

'''IS USERNAME DOMAIN IN EMAIL?'''
df_analysis['username_in_email_soft'] = df_analysis.apply(
    lambda row: username_in_email_soft(row['username'], row['email']),
    axis=1
)

df_analysis['similar_domain_and_picture'] = df_analysis.apply(
    lambda row: f"{'Yes' if row['user_has_profile_picture'] else 'No'} Picture / " +
                f"{'Yes' if row['username_in_email_soft'] else 'No'} Username in Email",
    axis=1
)




'''LOG SCALED VARIABLES'''
df_analysis['log_scaled_post_likes'] = np.log(df_analysis['post_likes'])

df_analysis['log_scaled_posts_retweets'] = np.log(df_analysis['posts_retweets'])

df_analysis['log_scaled_age'] = np.log(df_analysis['age'])

df_analysis['log_scaled_followers'] = np.log(df_analysis['followers'])

df_analysis['log_scaled_previous_posts_count'] = np.log(df_analysis['previous_posts_count'])

df_analysis['log_scaled_account_seniority_weeks'] = np.log(df_analysis['account_seniority_weeks'])

df_analysis['log_scaled_followers_to_previous_posts'] = np.log(df_analysis['followers'] / df_analysis['previous_posts_count'])

df_analysis['log_scaled_likes_to_retweets'] = np.log(df_analysis['post_likes'] / df_analysis['posts_retweets'])

df_analysis['log_scaled_likes_to_followers'] = np.log(df_analysis['post_likes'] / df_analysis['followers'])

df_analysis['log_scaled_word_count'] = np.log(df_analysis['text_word_count'])

df_analysis['log_scaled_hashtag_count'] = np.log(df_analysis['hashtag_count'])

df_analysis['log_scaled_posts_per_week'] = np.log(df_analysis['posts_per_week'])



"""### $\textbf{ניתוח התוכן של הטקסט}$"""

# chosen words and bi-grams after text analysis (analysis code in previous part)


top_words = ['feel','feeling','like','know','love','people','one',
             'much','want','think','life','something','still',
             'little','make','even','things','dont','bit']

top_bigrams = ['feel like','feeling like','makes feel','feel blessed','feel passionate','feeling generous','make feel',
               'still feel','want feel','feeling pretty','made feel','feel pretty','feel accepted','feeling little',
               'feel free', 'dont know','feel little','feeling bit','still feeling','help feel','feel bit','feel helpless']

for word in top_words:
    df_analysis[f'has_{word}'] = df['text'].apply(lambda x: int(word in x))

for bigram in top_bigrams:
    df_analysis[f'has_{bigram}'] = df['text'].apply(lambda x: int(bigram in x))


'''FEATURE REPRESENTATION'''


df_features = df_analysis.copy()
df_features = df_features.drop(columns=['text', 'id','username','email', 'post_datetime', 'timezone', 'embedded_content_url', 'profile_picture', 'account_creation_date', 'birthday','localized_post_datetime'])

df_features = df_features.loc[df_features['day_of_week'].notna()]


categorical_columns = df_features.select_dtypes(exclude=['number']).columns
numerical_columns = df_features.select_dtypes(exclude=['object', 'string', 'bool']).columns

# for Categorical Columns (One-Hot Encoding)
df_features = pd.get_dummies(df_features, columns=categorical_columns)

# for Numerical Columns (Min-Max Scaling)
scaler = MinMaxScaler()

#df_features['followers_to_previous_posts'] = df_features['followers_to_previous_posts'].dropna()
df_features[numerical_columns] = df_features[numerical_columns].replace([np.inf, -np.inf], np.nan)

# Iterate over each numerical column
for col in numerical_columns:
    max_value = df_features[col].max()  # Get the maximum value for the column
    df_features[col] = df_features[col].fillna(max_value)  # Fill missing values with the max value


df_features[numerical_columns] = scaler.fit_transform(df_features[numerical_columns])

df_training = df_features.copy()

# Create a new column: 1 if sentiment_positive is 1, 0 if sentiment_negative is 1
df_training['sentiment_bool'] = df_training.apply(
    lambda row: 1 if row['sentiment_positive'] == 1 else 0,
    axis=1
)

# deleting the dummies that were created for the sentiment category
df_training = df_training.drop(columns=['sentiment_positive', 'sentiment_negative'])

# chosen features from part 1 according the fisher score
df_training = df_training[['username_in_email_soft_True',
                                         'similar_domain_and_picture_Yes Picture / Yes Username in Email',
                                         'log_scaled_posts_per_week', 'hashtag_count',
                                         'log_scaled_likes_to_followers',
                                         'similar_domain_and_picture_Yes Picture / No Username in Email',
                                         'log_scaled_followers_to_previous_posts',
                                         'log_scaled_likes_to_retweets', 'log_scaled_previous_posts_count',
                                         'log_scaled_followers', 'month_of_post_December',
                                         'month_of_post_November', 'month_of_post_January',
                                         'month_of_post_February', 'checkmark_grey', 'checkmark_gold',
                                         'profile_picture_and_checkmark_status_1_gold',
                                         'profile_picture_and_checkmark_status_1_grey', 'checkmark_none',
                                         'profile_picture_and_checkmark_status_0_none', 'month_of_post_June',
                                         'localized_slot_of_day_Morning','sentiment_bool']]



df_training['sentiment_label'] = df_training['sentiment_bool'].map({1: 'positive', 0: 'negative'})
df_training = df_training.drop(columns=['sentiment_bool'])



In [ ]:
#------------------------------------------------------ Part 2 ------------------------------------------------------#
from sklearn.tree import DecisionTreeClassifier, plot_tree # for training and ploting DT
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split # to ensure similar sentiment distribution among the validation and training set
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, log_loss # for evaluation of the model using roc auc metric
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
import joblib # random forest





"""### $\textbf{סעיף 0}$"""

# defining the folds - 10 folds
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
x = df_training.drop(columns=['sentiment_label'])
y = df_training['sentiment_label']


"""### $\textbf{סעיף 1}$"""

"""### $\textbf{סעיף 1.1 - עץ החלטה}$"""

#---------------- arbitrary tree for preliminary visualization ----------------#

# storing results
auc_scores_val = []
auc_scores_train = []

# 10-fold cross-validation
for train_index, val_index in skf.split(x, y):

    X_train, X_val = x.iloc[train_index], x.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # re-scaling the training set to prevent data leakage from validation set
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)  # fit only on training set so it is not exposed to data from training set
    X_val_scaled = scaler.transform(X_val)          # transform validation with same scaler

    # training the model according to the entropy criterion
    model = DecisionTreeClassifier(
        criterion='entropy',
        max_depth=5,
        class_weight='balanced',
        random_state=42
    )
    model.fit(X_train_scaled, y_train)

    # predicting the probability of class 1 (positive class)
    y_train_probs = model.predict_proba(X_train_scaled)[:, 1]
    y_val_probs = model.predict_proba(X_val_scaled)[:, 1]

    # computing AUC on the training and validation set
    train_auc = roc_auc_score(y_train, y_train_probs)
    val_auc = roc_auc_score(y_val, y_val_probs)

    auc_scores_train.append(train_auc)
    auc_scores_val.append(val_auc)

# plotting an initial arbitrary tree
plt.figure(figsize=(45, 15))
plot_tree(
    model,
    filled=True,
    feature_names=x.columns.tolist(),
    rounded=True,
    fontsize=18,
    max_depth = 3
)
plt.tight_layout()
plt.show()


print("Average AUC Score for Validation Set:", sum(auc_scores_val) / len(auc_scores_val))
print("Average AUC Score for Training Set:", sum(auc_scores_train) / len(auc_scores_train))

#---------------- hyper parameter tuning for DT parameters ----------------#

# initializing a list of optional depths for the tree
max_depth_list = np.arange(1, 21, 1)

params_dt = {
    "clf__criterion": ["gini", "entropy"],
    "clf__max_depth": max_depth_list,
    "clf__min_samples_split": np.arange(100, 1001, 50),
    "clf__class_weight": [None, "balanced"]
}

# as discussed in question 0
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# ensures that the training set is re-scaled within the grid search
pipeline_dt = Pipeline([
    ('scaler', MinMaxScaler()),  # scales only within each fold
    ('clf', DecisionTreeClassifier(random_state=42))
])

grid = GridSearchCV(
    estimator=pipeline_dt,
    param_grid=params_dt,
    refit=True,
    scoring='roc_auc',
    verbose=3,
    cv=cv_strategy,
    return_train_score=True
)


# searching and finding the best model
grid.fit(x, y)


# grid search results
dt_results_df = pd.DataFrame(grid.cv_results_)
dt_results_df = dt_results_df.sort_values(by='mean_test_score', ascending=False)

dt_results_df[[
    'param_clf__criterion',
    'param_clf__class_weight',
    'param_clf__max_depth',
    'param_clf__min_samples_split',
    'mean_test_score',
    'rank_test_score'
]]

best_params_dt = grid.best_params_
print("Best Parameters Found by Grid Search:")
for param, val in best_params_dt.items():
    print(f"{param}: {val}")

# saving the best model according to the grid search
best_pipeline_dt = grid.best_estimator_
best_tree = best_pipeline_dt.named_steps['clf']

# getting the best model's index from GridSearchCV
best_idx = grid.best_index_

# creating DataFrame with per-fold AUC scores
folds_grid = {
    'Fold': list(range(1, 11)),
    'Train AUC': [dt_results_df[f'split{i}_train_score'][best_idx] for i in range(10)],
    'Validation AUC': [dt_results_df[f'split{i}_test_score'][best_idx] for i in range(10)]
}

folds_df = pd.DataFrame(folds_grid)

# printing table
print("AUC Scores per Fold:")
print(folds_df)

# Calculating means

train_scores = folds_df['Train AUC']
val_scores = folds_df['Validation AUC']

print("Average AUC Scores:")
print(f"Mean AUC on Training Set:   {np.mean(train_scores):.4f}")
print(f"Mean AUC on Validation Set: {np.mean(val_scores):.4f}") # same as best_tree.best_score_

# plotting the decision tree
plt.figure(figsize=(65, 30))
plot_tree(best_tree, filled=True, feature_names=x.columns.to_list(), max_depth = 3, fontsize=20)
plt.show()


# feature importance
feature_names = x.columns
importances = best_tree.feature_importances_

# matching feature to its importance
feature_importance_dict = dict(zip(feature_names, importances))

# sorting the importances
sorted_importances = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)

print("\nFeature Importances:")
for name, score in sorted_importances:
    print(f"{name}: {score:.4f}")

# plotting the features according to their importances

top_n = 10
top_features = sorted_importances[:top_n]
names, scores = zip(*top_features)

plt.figure(figsize=(8, 5))
plt.barh(names[::-1], scores[::-1])
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()




In [ ]:
"""### $\textbf{סעיף 1.2 - רשתות נוירונים מלאכותיות (MLP)}$"""

"""ANN"""


# יצירת pipeline
pipeline = make_pipeline(
    MinMaxScaler(),
    MLPClassifier(random_state=42, max_iter=100)
)

# אסטרטגיית Cross-Validation
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# אימון עם cross_validate כדי לקבל גם train וגם test scores
cv_results = cross_validate(
    pipeline,
    x, y,
    cv=cv_strategy,
    scoring='roc_auc',
    return_train_score=True
)

# טבלת תוצאות לפי folds
folds_grid = {
    'Fold': list(range(1, 11)),
    'Train AUC': cv_results['train_score'],
    'Validation AUC': cv_results['test_score']
}
folds_df = pd.DataFrame(folds_grid)

# הדפסת הטבלה
print("AUC Scores per Fold (Default Parameters):")
print(folds_df)

# ממוצעים
print("\nAverage AUC Scores:")
print(f"Mean AUC on Training Set:   {folds_df['Train AUC'].mean():.4f}")
print(f"Mean AUC on Validation Set: {folds_df['Validation AUC'].mean():.4f}")

# pipeline
pipeline = make_pipeline(
    MinMaxScaler(),
    MLPClassifier(max_iter=100, random_state=42)
)

# טווחי היפרפרמטרים
param_grid = {
    'mlpclassifier__hidden_layer_sizes': [(100,), (50,), (100, 50)],
    'mlpclassifier__learning_rate_init': [0.001, 0.01],
    'mlpclassifier__activation': ['relu', 'tanh']
}

cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# GridSearchCV עם AUC
grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring='roc_auc',
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1,
    return_train_score=True
)

# אימון
grid.fit(x, y)

# תצוגת סיכום של כל הקונפיגורציות
results = pd.DataFrame(grid.cv_results_)
summary = results[[
    'param_mlpclassifier__hidden_layer_sizes',
    'param_mlpclassifier__learning_rate_init',
    'param_mlpclassifier__activation',
    'mean_test_score',
    'rank_test_score'
]]
summary.columns = ['hidden_layer_sizes', 'learning_rate_init', 'activation', 'val_auc', 'rank']
summary = summary.sort_values(by='rank')

print("Grid Search Summary:")
print(summary.to_string(index=False))

# ----- חלק חדש: טבלת דיוק על כל Fold -----

# מציאת האינדקס של המודל הכי טוב
best_idx = grid.best_index_

# טבלת תוצאות לכל Fold
folds_grid = {
    'Fold': list(range(1, 11)),
    'Train AUC': [results[f'split{i}_train_score'][best_idx] for i in range(10)],
    'Validation AUC': [results[f'split{i}_test_score'][best_idx] for i in range(10)]
}
folds_df = pd.DataFrame(folds_grid)

print("\nAUC Scores per Fold (Best ANN Model):")
print(folds_df)

# ממוצעים
print("\nAverage AUC Scores:")
print(f"Mean AUC on Training Set:   {folds_df['Train AUC'].mean():.4f}")
print(f"Mean AUC on Validation Set: {folds_df['Validation AUC'].mean():.4f}")

"""גרף"""



# פיצול ל-Train ו-Validation
X_train, X_val, y_train, y_val = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

# נרמול
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# הגדרת המודל עם warm_start
model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation='tanh',
    learning_rate_init=0.001,
    max_iter=1,
    warm_start=True,
    random_state=42
)

# אחסון תוצאות לכל אפוק
train_loss = []
val_loss = []
train_auc = []
val_auc = []

# לולאת אימון ידנית ל־100 אפוקים
for epoch in range(100):
    model.fit(X_train_scaled, y_train)

    # Loss
    train_pred = model.predict_proba(X_train_scaled)[:, 1]
    val_pred = model.predict_proba(X_val_scaled)[:, 1]
    train_loss.append(log_loss(y_train, train_pred))
    val_loss.append(log_loss(y_val, val_pred))

    # AUC
    train_auc.append(roc_auc_score(y_train, train_pred))
    val_auc.append(roc_auc_score(y_val, val_pred))

# יצירת שני גרפים נפרדים
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# גרף דיוק (AUC)
ax1.plot(train_auc, label='train', color='blue')
ax1.plot(val_auc, label='valid', color='orange')
ax1.set_title('Model AUC')
ax1.set_xlabel('epoch')
ax1.set_ylabel('AUC')
ax1.legend()

# גרף Loss
ax2.plot(train_loss, label='train', color='blue')
ax2.plot(val_loss, label='valid', color='orange')
ax2.set_title('Model Loss')
ax2.set_xlabel('epoch')
ax2.set_ylabel('loss')
ax2.legend()

plt.tight_layout()
plt.show()



In [ ]:
"""### $\textbf{סעיף 1.3 - SVM}$"""

# re-defining to make sure it refers to the proper data and ot perhaps manipulated data
x = df_training.drop(columns=['sentiment_label'])
y = df_training['sentiment_label']

# הגדרת Pipeline
pipeline = make_pipeline(
    MinMaxScaler(),
    LinearSVC(max_iter=10000)
)

# טווח היפרפרמטרים
param_grid = {
    'linearsvc__C': [0.01, 0.1, 1, 10, 100, 1000],
    'linearsvc__loss': ['hinge', 'squared_hinge']
}

# אסטרטגיית K-Fold
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# GridSearchCV עם החזרת train score
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    verbose=1,
    n_jobs=-1,
    return_train_score=True
)

# אימון
grid.fit(x, y)

# תוצאות כלליות
print("Best parameters:", grid.best_params_)

# DataFrame של התוצאות
svm_results_df = pd.DataFrame(grid.cv_results_)

# אינדקס של המודל הכי טוב
best_idx = grid.best_index_

# יצירת טבלה של AUC לפי קיפולים
folds_grid = {
    'Fold': list(range(1, 11)),
    'Train AUC': [svm_results_df[f'split{i}_train_score'][best_idx] for i in range(10)],
    'Validation AUC': [svm_results_df[f'split{i}_test_score'][best_idx] for i in range(10)]
}
folds_df = pd.DataFrame(folds_grid)

# הדפסת טבלה
print("\nAUC Scores per Fold:")
print(folds_df)

# ממוצעים
train_scores = folds_df['Train AUC']
val_scores = folds_df['Validation AUC']

print("\nAverage AUC Scores:")
print(f"Mean AUC on Training Set:   {np.mean(train_scores):.4f}")
print(f"Mean AUC on Validation Set: {np.mean(val_scores):.4f}")

"""משוואת הישר"""

# נשלוף את המודל מתוך הפייפליין
best_model = grid.best_estimator_
svm_model = best_model.named_steps['linearsvc']

# וקטור המקדמים (למודל בינארי יש רק שורה אחת)
coefs = svm_model.coef_[0]
intercept = svm_model.intercept_[0]

# התאמה בין שמות הפיצ’רים למקדמים
for feature, weight in zip(x.columns, coefs):
    print(f"{feature}: {weight:.4f}")

print(f"Intercept (bias): {intercept:.4f}")

In [ ]:
"""### $\textbf{סעיף 1.4 - אשכול}$"""

# procedures due to issues with KMedoids and gower libraries
!pip uninstall -y scikit-learn-extra numpy
!pip install --no-cache-dir numpy==1.23.5
!pip install --no-cache-dir scikit-learn-extra

!pip install -q scikit-learn-extra
!pip install -q gower

from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score
import gower


features_for_clustering = [
    'log_scaled_posts_per_week',
    'log_scaled_likes_to_followers',
    'log_scaled_followers_to_previous_posts',
    'similar_domain_and_picture_Yes Picture / No Username in Email',
    'localized_slot_of_day_Morning'
]


df_cluster = df_training[features_for_clustering]

# visualization of pairplot
numeric_cols = ds.select_dtypes(include=['number']).columns
cols_to_plot = numeric_cols.tolist() + ['sentiment_label']
sns.pairplot(ds[cols_to_plot], hue='sentiment_label', corner =True)


# clustering on a subset of features from dataset
df_sample = df_cluster.sample(n=4000, random_state=42)

gower_matrix = gower.gower_matrix(df_sample)

k_range = list(range(2, 7))
silhouette_scores = []

for k in k_range:
    model = KMedoids(n_clusters=k, metric='precomputed', max_iter=300, random_state=42)
    labels = model.fit_predict(gower_matrix)
    sil_score = silhouette_score(gower_matrix, labels, metric='precomputed')
    silhouette_scores.append(sil_score)
    print(f"K={k} --> Silhouette Score = {sil_score:.4f}")


# plotting results for sub-set of features
plt.figure(figsize=(6, 4))
plt.plot(k_range, silhouette_scores, marker='o')
plt.title('Silhouette Score by Number of Clusters (K)')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()


# clustering on all of the features of the dataset
df_sample = ds.sample(n=4000, random_state=42)

gower_matrix = gower.gower_matrix(df_sample)

k_range = list(range(2, 7))
silhouette_scores = []

for k in k_range:
    model = KMedoids(n_clusters=k, metric='precomputed', max_iter=300, random_state=42)
    labels = model.fit_predict(gower_matrix)
    sil_score = silhouette_score(gower_matrix, labels, metric='precomputed')
    silhouette_scores.append(sil_score)
    print(f"K={k} --> Silhouette Score = {sil_score:.4f}")


plt.figure(figsize=(6, 4))
plt.plot(k_range, silhouette_scores, marker='o')
plt.title('Silhouette Score by Number of Clusters (K) on All Dataset Features')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()


# only numerical columns for PCA
X_cluster = ds[cols_to_plot]

# taking a random sample of 4000 rows from the dataset
X_cluster = X_cluster.sample(n=4000, random_state=42)
sentiment_labels = X_cluster['sentiment_label']

# removing the 'sentiment_label' column from the sample DataFrame
X_cluster = X_cluster.drop('sentiment_label', axis=1)

# computing the Gower distance matrix
gower_matrix = gower.gower_matrix(df_sample)

# initializing K-Medoids with precomputed Gower distances
kmedoids = KMedoids(n_clusters=2, metric='precomputed', max_iter=300, random_state=42)

# applying PCA to reduce the data to 2 components (for visualization)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster)

# Creating a DataFrame from the PCA results
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])

# fitting K-Medoids on the Gower distance matrix and getting cluster labels
cluster_labels = kmedoids.fit_predict(gower_matrix)

# adding cluster labels to the PCA DataFrame
pca_df['Cluster'] = cluster_labels
pca_df['Sentiment'] = sentiment_labels.values

# plotting clustering results with PCA
plt.figure(figsize=(14, 6))

# Left: Cluster label
plt.subplot(1, 2, 1)
scatter1 = plt.scatter(pca_df['PC1'], pca_df['PC2'], c=pca_df['Cluster'], cmap='Paired', s=30)
plt.title('K-Medoids Clusters (PCA)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(scatter1, label='Cluster')

# Right: Sentiment label
plt.subplot(1, 2, 2)
scatter2 = plt.scatter(pca_df['PC1'], pca_df['PC2'], c=pd.factorize(pca_df['Sentiment'])[0], cmap='coolwarm', s=30)
plt.title('Sentiment Labels (PCA)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(scatter2, label='Sentiment')

plt.tight_layout()
plt.show()



In [ ]:
"""### $\textbf{סעיף 2 - הערכה (אבליואציה)}$"""

# re-defining to make sure it refers to the proper data and ot perhaps manipulated data
x = df_training.drop(columns=['sentiment_label'])
y = df_training['sentiment_label']
y_binary = (y == 'positive').astype(int) # for prediction purposes

# re-training the decision tree after hyper-parameter tuning
# runtime was interrupted so we needed to re-train the model (sorry)
pipeline_dt = Pipeline([
    ('scaler', MinMaxScaler()),
    ('clf', DecisionTreeClassifier(
        criterion='entropy',
        max_depth=14,
        min_samples_split=250,
        class_weight=None,
        random_state=42
    ))
])

# colors for consistent visualization
colors = plt.cm.tab10(np.linspace(0, 1, 10))

# storage for all results
fold_results = []
cumulative_cm = np.zeros((2, 2), dtype=int)

# create figure with subplots
fig = plt.figure(figsize=(18, 12))

# ROC Curves subplot
ax1 = plt.subplot(2, 2, 1)

# looping over folds
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(x, y_binary)):

    X_train, X_val = x.iloc[train_idx], x.iloc[val_idx]
    y_train, y_val = y_binary.iloc[train_idx], y_binary.iloc[val_idx]

    # training pipeline
    pipeline_dt.fit(X_train, y_train)

    # getting probabilities
    y_train_probs = pipeline_dt.predict_proba(X_train)[:, 1]
    y_val_probs = pipeline_dt.predict_proba(X_val)[:, 1]

    # ROC curve for training (for threshold selection)
    fpr_train, tpr_train, thresholds_train = roc_curve(y_train, y_train_probs)

    # ROC curve for validation (for plotting)
    fpr_val, tpr_val, thresholds_val = roc_curve(y_val, y_val_probs)
    val_auc = auc(fpr_val, tpr_val)

    # selecting threshold from training set (specificity >= 95%)
    specificity_train = 1 - fpr_train
    min_specificity = 0.95
    valid_indices = specificity_train >= min_specificity

    if np.any(valid_indices):
        valid_tpr = tpr_train[valid_indices]
        valid_thresholds = thresholds_train[valid_indices]
        best_threshold = valid_thresholds[np.argmax(valid_tpr)]
    else:
        best_threshold = thresholds_train[np.argmax(specificity_train)]

    # applying threshold to validation set
    y_val_pred = (y_val_probs >= best_threshold).astype(int)

    # calculating confusion matrix
    fold_cm = confusion_matrix(y_val, y_val_pred)
    cumulative_cm += fold_cm

    # calculating actual metrics from confusion matrix
    tn, fp, fn, tp = fold_cm.ravel()
    actual_specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    actual_sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    actual_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    actual_accuracy = (tp + tn) / (tp + tn + fp + fn)

    # calculating FPR and TPR for the selected threshold point
    actual_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    actual_tpr = tp / (tp + fn) if (tp + fn) > 0 else 0

    # storing results
    fold_results.append({
        'fold': fold_idx + 1,
        'threshold': best_threshold,
        'val_auc': val_auc,
        'specificity': actual_specificity,
        'sensitivity': actual_sensitivity,
        'precision': actual_precision,
        'accuracy': actual_accuracy,
        'fpr': actual_fpr,
        'tpr': actual_tpr,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp
    })

    # plotting validation ROC curve
    ax1.plot(fpr_val, tpr_val, color=colors[fold_idx], alpha=0.8, linewidth=2,
             label=f'Fold {fold_idx + 1} (AUC = {val_auc:.4f})')

    # plotting the actual point from confusion matrix
    offset_x = 0.005 * (fold_idx % 3 - 1)
    offset_y = 0.005 * ((fold_idx // 3) % 3 - 1)
    ax1.scatter(actual_fpr + offset_x, actual_tpr + offset_y,
               color=colors[fold_idx], s=80, edgecolor='black', linewidth=1.5, zorder=5)

# finalizing ROC plot
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.6, linewidth=1, label='Random classifier')
ax1.set_xlim([-0.02, 1.02])
ax1.set_ylim([-0.02, 1.02])
ax1.set_xlabel('False Positive Rate (FPR)', fontsize=12, fontweight='bold')
ax1.set_ylabel('True Positive Rate (TPR)', fontsize=12, fontweight='bold')
ax1.set_title('ROC Curves with Selected Thresholds', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=8)
ax1.grid(True, alpha=0.3)

# Confusion Matrix - Absolute numbers
ax2 = plt.subplot(2, 2, 2)
sns.heatmap(cumulative_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'],
            ax=ax2, cbar_kws={'label': 'Count'})
ax2.set_title('Cumulative Confusion Matrix\n(Absolute Numbers)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax2.set_ylabel('True Label', fontsize=12, fontweight='bold')


# Confusion Matrix - Normalized
ax3 = plt.subplot(2, 2, 3)
cm_normalized = cumulative_cm.astype('float') / cumulative_cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Greens',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'],
            ax=ax3, cbar_kws={'label': 'Proportion'})
ax3.set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold')
ax3.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax3.set_ylabel('True Label', fontsize=12, fontweight='bold')

# Metrics comparison plot
ax4 = plt.subplot(2, 2, 4)
folds = [r['fold'] for r in fold_results]
specificities = [r['specificity'] for r in fold_results]
sensitivities = [r['sensitivity'] for r in fold_results]
aucs = [r['val_auc'] for r in fold_results]

ax4.plot(folds, specificities, 'o-', color='blue', linewidth=2, markersize=6, label='Specificity')
ax4.plot(folds, sensitivities, 's-', color='red', linewidth=2, markersize=6, label='Sensitivity')
ax4.plot(folds, aucs, '^-', color='green', linewidth=2, markersize=6, label='AUC')
ax4.axhline(y=0.95, color='blue', linestyle='--', alpha=0.7, label='95% Specificity Target')
ax4.set_xlabel('Fold', fontsize=12, fontweight='bold')
ax4.set_ylabel('Score', fontsize=12, fontweight='bold')
ax4.set_title('Metrics per Fold', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_xticks(range(1, 11, 1))
ax4.set_ylim(0.9, 1.0)
ax4.set_yticks(np.arange(0.90, 1.01, 0.01))


plt.tight_layout()
plt.show()

# printing detailed results
print("\n" + "="*110)
print("DETAILED RESULTS")
print("="*110)
print("Fold\tAUC\tAccuracy\tSpecificity\tSensitivity\tPrecision\tThreshold\tFPR\tTPR")
print("-" * 110)

for result in fold_results:
    print(f"{result['fold']}\t{result['val_auc']:.4f}\t{result['accuracy']:.4f}\t\t"
          f"{result['specificity']:.4f}\t\t{result['sensitivity']:.4f}\t\t"
          f"{result['precision']:.4f}\t\t{result['threshold']:.6f}\t{result['fpr']:.4f}\t{result['tpr']:.4f}")

# Summary statistics
aucs = [r['val_auc'] for r in fold_results]
specificities = [r['specificity'] for r in fold_results]
sensitivities = [r['sensitivity'] for r in fold_results]
accuracies = [r['accuracy'] for r in fold_results]
fprs = [r['fpr'] for r in fold_results]
tprs = [r['tpr'] for r in fold_results]
thresholds = [r['threshold'] for r in fold_results]



print("\n" + "-"*110)
print("SUMMARY STATISTICS")
print("-"*110)
print(f"Validation AUC: Mean={np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"Specificity: Mean={np.mean(specificities):.4f} ± {np.std(specificities):.4f}")
print(f"Sensitivity: Mean={np.mean(sensitivities):.4f} ± {np.std(sensitivities):.4f}")
print(f"Accuracy:    Mean={np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"FPR:         Mean={np.mean(fprs):.4f} ± {np.std(fprs):.4f}")
print(f"TPR:         Mean={np.mean(tprs):.4f} ± {np.std(tprs):.4f}")
print(f"Threshold:   Mean={np.mean(thresholds):.4f} ± {np.std(thresholds):.4f}")



# calculating overall metrics from cumulative confusion matrix
tn_total, fp_total, fn_total, tp_total = cumulative_cm.ravel()
overall_specificity = tn_total / (tn_total + fp_total)
overall_sensitivity = tp_total / (tp_total + fn_total)
overall_accuracy = (tp_total + tn_total) / cumulative_cm.sum()



In [ ]:
"""### $\textbf{סעיף 3 - שיפור המודל)}$"""

"""Improvements"""
# applying the same data processing as before except the mentioned improvements
# uploading the dataset
uploaded = files.upload()
df = pd.read_csv("sentiment.csv")

# creating an auxilliary df that stores all of the columns as strings for easier filtering
# removing "?" from the target variable column
df_str = df.astype(str)
df_str = df_str[df_str['sentiment'] != '?']
'''PRE-PROCESSING'''

df_analysis = df.copy()

df_analysis = df_analysis.astype('string')

'''PRE-PROCESSING - CHECKING FOR QUESTION MARKS (MISSING VALUES)'''

rows_with_question_marks = df.apply(lambda row: row.str.contains('\?').any(), axis=1).sum()

print(f"Number of rows with at least one question mark: {rows_with_question_marks}")

df_clean_target = df[df['sentiment'] != "?"]


'''REMOVING OUTLIERS USING THE IQR METHOD'''

def remove_outliers_iqr(df):
    """
    Removes outliers from all numerical columns in the DataFrame using the IQR method.

    Parameters:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A new DataFrame with outliers removed.
    """
    df_clean = df.copy()
    numeric_cols = df_clean.select_dtypes(include='number').columns

    for col in numeric_cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        mask = (df_clean[col] >= Q1 - 1.5 * IQR) & (df_clean[col] <= Q3 + 1.5 * IQR)
        df_clean = df_clean[mask]

    return df_clean

'''FILTERED DATAFRAME'''

df_analysis = remove_outliers_iqr(df_analysis)


# Count number of '?' per row
missing_counts = (df_analysis == "?").sum(axis=1)

# Keep only rows with fewer than 8 question marks
df_analysis = df_analysis[missing_counts < 8]

'''PRE-PROCESSING - HANDLING MISSING VALUES PER COLUMN'''
df_analysis = df_analysis[df_analysis['sentiment'] != "?"]

def fill_categorical_missing_by_probability_columns(df, columns):
    if isinstance(columns, str):
        columns = [columns]  # if a single column is passed as a string, make it a list

    for column in columns:
        # Replace "?" with np.nan first
        df[column] = df[column].astype(str)
        df[column] = df[column].replace("?", np.nan)

        # Get non-missing values
        non_missing = df[column].dropna()

        if non_missing.empty:
            continue  # skip if no non-missing values to sample from

        # Calculate probabilities for each category
        value_counts = non_missing.value_counts(normalize=True)

        # Find missing indexes (rows) that need to be filled
        missing_idx = df[df[column].isna()].index

        if len(missing_idx) == 0:
            continue  # skip if there are no missing values

        # Randomly sample based on probabilities
        sampled_values = np.random.choice(value_counts.index, size=len(missing_idx), p=value_counts.values)

        # Fill missing
        df.loc[missing_idx, column] = sampled_values

    return df

def fill_numerical_missing_columns(df, columns):
    if isinstance(columns, str):
        columns = [columns]

    # Convert '?' to np.nan and cast to numeric
    for col in columns:
        df[col] = df[col].astype(str).replace("?", np.nan)
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Apply KNN imputation on the selected numeric columns
    imputer = KNNImputer(n_neighbors=5)
    imputed_values = imputer.fit_transform(df[columns])

    # Replace the original columns with the imputed values
    df[columns] = imputed_values

    return df


df_analysis = fill_numerical_missing_columns(df_analysis, ['post_likes','posts_retweets','followers','previous_posts_count'])

df_analysis = fill_categorical_missing_by_probability_columns(df_analysis, ['type', 'checkmark', 'timezone'])


# converting numerical columns to numeric data type
df_analysis['post_likes'] = pd.to_numeric(df_analysis['post_likes'], errors='coerce')
df_analysis['posts_retweets'] = pd.to_numeric(df_analysis['posts_retweets'], errors='coerce')
df_analysis['followers'] = pd.to_numeric(df_analysis['followers'], errors='coerce')
df_analysis['previous_posts_count'] = pd.to_numeric(df_analysis['previous_posts_count'], errors='coerce')

# setting datetime-type columns
df_analysis['post_datetime'] = pd.to_datetime(df_analysis['post_datetime'], errors='coerce')
df_analysis['account_creation_date'] = pd.to_datetime(df_analysis['account_creation_date'], errors='coerce')
df_analysis['birthday'] = pd.to_datetime(df_analysis['birthday'], errors='coerce')

'''PRE-PROCESSING - CONVERTING DATA TYPES AND FORMATTING'''

'''MONTH OF POST'''
df_analysis['month_of_post'] = df_analysis['post_datetime'].dt.month_name() # January, February...

from zoneinfo import ZoneInfo
'''LOCALIZED POST DATETIME'''
df_analysis['localized_post_datetime'] = [
    localize_post_time(post_time, timezone)
    for post_time, timezone in zip(df_analysis['post_datetime'], df_analysis['timezone'])
]

'''MANIPULATING VARIABLES AFTER PRE-PROCESSING'''
'''ASSUMPTION: ALL OF THE DATA TYPES ARE COORDINATED AND ALIGNED'''

'''REGION'''
df_analysis['region'] = df_analysis['timezone'].map(timezone_to_region)
df_analysis['region'] = df_analysis['region'].astype('string')


'''DAY OF WEEK'''
df_analysis['day_of_week'] = df_analysis['post_datetime'].dt.day_name()
df_analysis['day_of_week'] = df_analysis['day_of_week'].astype('string')


'''AGE'''
# transform birthday to age at post using apply
df_analysis['age'] = df_analysis.apply(lambda row: calculate_age(row['birthday'], row['localized_post_datetime']), axis=1)
df_analysis['age'] = pd.to_numeric(df_analysis['age'])

'''LOCALIZED SLOT OF DAY'''
df_analysis['localized_slot_of_day'] = df_analysis['localized_post_datetime'].apply(get_slot_of_day)
df_analysis['localized_slot_of_day'] = df_analysis['localized_slot_of_day'].astype('string')


'''DAILY POST RETWEET RATE'''
df_analysis['daily_retweet_rate'] = df_analysis['posts_retweets'] / 30
df_analysis['daily_retweet_rate'] = pd.to_numeric(df_analysis['daily_retweet_rate'])



'''ACCOUNT SENIORITY (AGE AT POST) IN WEEKS'''
# calculating account seniority in weeks according to post date
df_analysis['account_seniority_days'] = (df_analysis['post_datetime'] - df_analysis['account_creation_date']).dt.days
df_analysis['account_seniority_days'] = pd.to_numeric(df_analysis['account_seniority_days'])

df_analysis['account_seniority_weeks'] = df_analysis['account_seniority_days'] / 7
# replacing 0 weeks with 1 to avoid division by zero
df_analysis['account_seniority_weeks'] = df_analysis['account_seniority_weeks'].replace(0, 1)
df_analysis['account_seniority_weeks'] = pd.to_numeric(df_analysis['account_seniority_weeks'])


'''POSTS PER WEEK'''
# calculating posts per week
df_analysis['posts_per_week'] = df_analysis['previous_posts_count'] / df_analysis['account_seniority_weeks']
df_analysis['posts_per_week'] = pd.to_numeric(df_analysis['posts_per_week'])



'''FOLLOWERS PER WEEK'''
df_analysis['followers_per_week'] = df_analysis['followers'] / df_analysis['account_seniority_weeks']
df_analysis['followers_per_week'] = pd.to_numeric(df_analysis['followers_per_week'])


'''FOLLOWERS TO PREVIOUS POSTS RATE'''
df_analysis['followers_to_previous_posts'] = df_analysis['followers'] / df_analysis['previous_posts_count']
df_analysis['followers_to_previous_posts'] = pd.to_numeric(df_analysis['followers_to_previous_posts'])


'''LIKES TO RETWEETS RATE'''
df_analysis['likes_to_retweets_rate'] = (df_analysis['post_likes'] / df_analysis['posts_retweets'])
df_analysis['likes_to_retweets_rate'] = pd.to_numeric(df_analysis['likes_to_retweets_rate'])


'''LIKES TO FOLLOWERS RATE'''
df_analysis['likes_to_followers_rate'] = df_analysis['post_likes'] / df_analysis['followers']
df_analysis['likes_to_followers_rate'] = pd.to_numeric(df_analysis['likes_to_followers_rate'])


'''TEXT WORD COUNT'''
df_analysis['text_word_count'] = df_analysis['text'].apply(lambda x: len(x.split()))
df_analysis['text_word_count'] = pd.to_numeric(df_analysis['text_word_count'])


'''TEXT # COUNT'''
df_analysis['hashtag_count'] = df_analysis['text'].apply(lambda x: len([word for word in x.split() if word.startswith('#')]))
df_analysis['hashtag_count'] = pd.to_numeric(df_analysis['hashtag_count'])


'''ATTACHED CONTENT TYPE IN POST'''
df_analysis['attached_content_type'] = df_analysis['embedded_content_url'].apply(classify_and_get_content_type)
df_analysis['attached_content_type'] = df_analysis['attached_content_type'].astype('string')


'''USER HAS PROFILE PICTURE'''
df_analysis['user_has_profile_picture'] = df_analysis['profile_picture'].apply(
    lambda x: 0 if pd.isna(x) or x == "?" else 1
)
df_analysis['user_has_profile_picture'] = pd.to_numeric(df_analysis['user_has_profile_picture'])



'''COMBINED FEATURE FOR ATTACHED CONTENT TYPE AND PICTURE'''
df_analysis['attached_content_and_picture_status'] = df_analysis.apply(
    lambda row: f"{row['attached_content_type']}_with_pic" if row['user_has_profile_picture'] == 1 else f"{row['attached_content_type']}_no_pic",
    axis=1
)
df_analysis['attached_content_and_picture_status'] = df_analysis['attached_content_and_picture_status'].astype('string')


'''COMBINED FEATURE FOR CHECKMARK (TYPE OF ACCOUNT) AND PROFILE PICTURE'''
df_analysis['profile_picture_and_checkmark_status'] = df_analysis.apply(
    lambda row: f"{row['user_has_profile_picture']}_{row['checkmark']}", axis=1)
df_analysis['profile_picture_and_checkmark_status'] = df_analysis['profile_picture_and_checkmark_status'].astype('string')


'''COMBINED FEATURE FOR PROFILE PICTURE AND USERNAME IN EMAIL'''

'''IS USERNAME DOMAIN IN EMAIL?'''
df_analysis['username_in_email_soft'] = df_analysis.apply(
    lambda row: username_in_email_soft(row['username'], row['email']),
    axis=1
)

df_analysis['similar_domain_and_picture'] = df_analysis.apply(
    lambda row: f"{'Yes' if row['user_has_profile_picture'] else 'No'} Picture / " +
                f"{'Yes' if row['username_in_email_soft'] else 'No'} Username in Email",
    axis=1
)




'''LOG SCALED VARIABLES'''
df_analysis['log_scaled_post_likes'] = np.log(df_analysis['post_likes'])

df_analysis['log_scaled_posts_retweets'] = np.log(df_analysis['posts_retweets'])

df_analysis['log_scaled_age'] = np.log(df_analysis['age'])

df_analysis['log_scaled_followers'] = np.log(df_analysis['followers'])

df_analysis['log_scaled_previous_posts_count'] = np.log(df_analysis['previous_posts_count'])

df_analysis['log_scaled_account_seniority_weeks'] = np.log(df_analysis['account_seniority_weeks'])

df_analysis['log_scaled_followers_to_previous_posts'] = np.log(df_analysis['followers'] / df_analysis['previous_posts_count'])

df_analysis['log_scaled_likes_to_retweets'] = np.log(df_analysis['post_likes'] / df_analysis['posts_retweets'])

df_analysis['log_scaled_likes_to_followers'] = np.log(df_analysis['post_likes'] / df_analysis['followers'])

df_analysis['log_scaled_word_count'] = np.log(df_analysis['text_word_count'])

df_analysis['log_scaled_hashtag_count'] = np.log(df_analysis['hashtag_count'])

df_analysis['log_scaled_posts_per_week'] = np.log(df_analysis['posts_per_week'])


top_words = ['feel','feeling','like','know','love','people','one',
             'much','want','think','life','something','still',
             'little','make','even','things','dont','bit']

top_bigrams = ['feel like','feeling like','makes feel','feel blessed','feel passionate','feeling generous','make feel',
               'still feel','want feel','feeling pretty','made feel','feel pretty','feel accepted','feeling little',
               'feel free', 'dont know','feel little','feeling bit','still feeling','help feel','feel bit','feel helpless']

for word in top_words:
    df_analysis[f'has_{word}'] = df['text'].apply(lambda x: int(word in x))

for bigram in top_bigrams:
    df_analysis[f'has_{bigram}'] = df['text'].apply(lambda x: int(bigram in x))

'''FEATURE REPRESENTATION'''

df_features = df_analysis.copy()
df_features = df_features.drop(columns=['text', 'id','username','email', 'post_datetime', 'timezone', 'embedded_content_url', 'profile_picture', 'account_creation_date', 'birthday','localized_post_datetime'])

df_features = df_features.loc[df_features['day_of_week'].notna()]


categorical_columns = df_features.select_dtypes(exclude=['number']).columns
numerical_columns = df_features.select_dtypes(exclude=['object', 'string', 'bool']).columns

# for Categorical Columns (One-Hot Encoding)
df_features = pd.get_dummies(df_features, columns=categorical_columns)

# for Numerical Columns (Min-Max Scaling)
scaler = MinMaxScaler()

#df_features['followers_to_previous_posts'] = df_features['followers_to_previous_posts'].dropna()
df_features[numerical_columns] = df_features[numerical_columns].replace([np.inf, -np.inf], np.nan)

# Iterate over each numerical column
for col in numerical_columns:
    max_value = df_features[col].max()  # Get the maximum value for the column
    df_features[col] = df_features[col].fillna(max_value)  # Fill missing values with the max value


df_features[numerical_columns] = scaler.fit_transform(df_features[numerical_columns])

df_training = df_features.copy()

# Create a new column: 1 if sentiment_positive is 1, 0 if sentiment_negative is 1
df_training['sentiment_bool'] = df_training.apply(
    lambda row: 1 if row['sentiment_positive'] == 1 else 0,
    axis=1
)

df_training = df_training.drop(columns=['sentiment_positive', 'sentiment_negative'])


df_training = df_training[['username_in_email_soft_True',
                                         'similar_domain_and_picture_Yes Picture / Yes Username in Email',
                                         'log_scaled_posts_per_week', 'hashtag_count',
                                         'log_scaled_likes_to_followers',
                                         'similar_domain_and_picture_Yes Picture / No Username in Email',
                                         'log_scaled_followers_to_previous_posts',
                                         'log_scaled_likes_to_retweets', 'log_scaled_previous_posts_count',
                                         'log_scaled_followers', 'month_of_post_December',
                                         'month_of_post_November', 'month_of_post_January',
                                         'month_of_post_February', 'checkmark_grey', 'checkmark_gold',
                                         'profile_picture_and_checkmark_status_1_gold',
                                         'profile_picture_and_checkmark_status_1_grey', 'checkmark_none',
                                         'profile_picture_and_checkmark_status_0_none', 'month_of_post_June',
                                         'localized_slot_of_day_Morning','sentiment_bool']]



"""random forest"""


# הגדרת x ו-y
x = df_training.drop(columns=['sentiment_bool'])
y = df_training['sentiment_bool']

# יצירת pipeline
pipeline = make_pipeline(
    MinMaxScaler(),
    RandomForestClassifier(random_state=42)
)

# טווחי היפרפרמטרים לחיפוש רנדומלי
param_distributions = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [None, 5, 10, 20],
    'randomforestclassifier__min_samples_split': [2, 5, 10],
    'randomforestclassifier__max_features': ['sqrt', 'log2']
}

# הגדרת קרוס־ולידציה מרובדת
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# RandomizedSearchCV
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=20,  # כמות הצירופים שיבדקו
    scoring='roc_auc',
    cv=cv_strategy,
    return_train_score=True,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

# הרצת החיפוש
search.fit(x, y)

# שליפת תוצאות לפי המודל הטוב ביותר
best_idx = search.best_index_
results_df = pd.DataFrame(search.cv_results_)

# שליפת התוצאות לכל fold
folds_data = {
    'Fold': list(range(1, 11)),
    'Train AUC': [results_df[f'split{i}_train_score'][best_idx] for i in range(10)],
    'Validation AUC': [results_df[f'split{i}_test_score'][best_idx] for i in range(10)]
}

folds_df = pd.DataFrame(folds_data)

# הדפסת טבלה
print("AUC Scores per Fold:")
print(folds_df)

# ממוצעים
mean_train_auc = np.mean(folds_df['Train AUC'])
mean_val_auc = np.mean(folds_df['Validation AUC'])

print("\nAverage AUC Scores:")
print(f"Train AUC: {mean_train_auc:.4f} ({mean_train_auc * 100:.2f}%)")
print(f"Validation AUC: {mean_val_auc:.4f} ({mean_val_auc * 100:.2f}%)")

# הדפסת ההיפרפרמטרים שנבחרו
print("\nBest Parameters:")
print(search.best_params_)

joblib.dump(search.best_estimator_, 'improved_best_model.pkl')



In [ ]:
"""### $\textbf{סעיף 4 - בדיקה (טסט)}$"""


# uploading the test set
uploaded = files.upload()
df_test = pd.read_csv("test.csv")

# creating an auxilliary df that stores all of the columns as strings for easier filtering
# removing "?" from the target variable column
df_test_str = df_test.astype(str)
'''PRE-PROCESSING'''

df_test_processed = df_test_str.copy()

'''PRE-PROCESSING - CHECKING FOR QUESTION MARKS (MISSING VALUES)'''

rows_with_question_marks = df_test_processed.apply(lambda row: row.str.contains('\?').any(), axis=1).sum()

print(f"Number of rows with at least one question mark: {rows_with_question_marks}")



'''REMOVING OUTLIERS USING THE IQR METHOD'''

def remove_outliers_iqr(df):
    """
    Removes outliers from all numerical columns in the DataFrame using the IQR method.

    Parameters:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A new DataFrame with outliers removed.
    """
    df_clean = df.copy()
    numeric_cols = df_clean.select_dtypes(include='number').columns

    for col in numeric_cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        mask = (df_clean[col] >= Q1 - 1.5 * IQR) & (df_clean[col] <= Q3 + 1.5 * IQR)
        df_clean = df_clean[mask]

    return df_clean

'''FILTERED DATAFRAME'''

df_test_processed = remove_outliers_iqr(df_test_processed)


# Count number of '?' per row
missing_counts = (df_test_processed == "?").sum(axis=1)

# Keep only rows with fewer than 8 question marks
df_test_processed = df_test_processed[missing_counts < 8]

'''PRE-PROCESSING - HANDLING MISSING VALUES PER COLUMN'''

def fill_categorical_missing_by_probability_columns(df, columns):
    if isinstance(columns, str):
        columns = [columns]  # if a single column is passed as a string, make it a list

    for column in columns:
        # Replace "?" with np.nan first
        df[column] = df[column].astype(str)
        df[column] = df[column].replace("?", np.nan)

        # Get non-missing values
        non_missing = df[column].dropna()

        if non_missing.empty:
            continue  # skip if no non-missing values to sample from

        # Calculate probabilities for each category
        value_counts = non_missing.value_counts(normalize=True)

        # Find missing indexes (rows) that need to be filled
        missing_idx = df[df[column].isna()].index

        if len(missing_idx) == 0:
            continue  # skip if there are no missing values

        # Randomly sample based on probabilities
        sampled_values = np.random.choice(value_counts.index, size=len(missing_idx), p=value_counts.values)

        # Fill missing
        df.loc[missing_idx, column] = sampled_values

    return df

def fill_numerical_missing_columns(df, columns):
    if isinstance(columns, str):
        columns = [columns]

    # Convert '?' to np.nan and cast to numeric
    for col in columns:
        df[col] = df[col].astype(str).replace("?", np.nan)
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Apply KNN imputation on the selected numeric columns
    imputer = KNNImputer(n_neighbors=5)
    imputed_values = imputer.fit_transform(df[columns])

    # Replace the original columns with the imputed values
    df[columns] = imputed_values

    return df


df_test_processed = fill_numerical_missing_columns(df_test_processed, ['post_likes','posts_retweets','followers','previous_posts_count'])

df_test_processed = fill_categorical_missing_by_probability_columns(df_test_processed, ['type', 'checkmark', 'timezone'])


# converting numerical columns to numeric data type
df_test_processed['post_likes'] = pd.to_numeric(df_test_processed['post_likes'], errors='coerce')
df_test_processed['posts_retweets'] = pd.to_numeric(df_test_processed['posts_retweets'], errors='coerce')
df_test_processed['followers'] = pd.to_numeric(df_test_processed['followers'], errors='coerce')
df_test_processed['previous_posts_count'] = pd.to_numeric(df_test_processed['previous_posts_count'], errors='coerce')

# setting datetime-type columns
df_test_processed['post_datetime'] = pd.to_datetime(df_test_processed['post_datetime'], errors='coerce')
df_test_processed['account_creation_date'] = pd.to_datetime(df_test_processed['account_creation_date'], errors='coerce')
df_test_processed['birthday'] = pd.to_datetime(df_test_processed['birthday'], errors='coerce')

'''PRE-PROCESSING - CONVERTING DATA TYPES AND FORMATTING'''

'''MONTH OF POST'''
df_test_processed['month_of_post'] = df_test_processed['post_datetime'].dt.month_name() # January, February...

from zoneinfo import ZoneInfo
'''LOCALIZED POST DATETIME'''
df_test_processed['localized_post_datetime'] = [
    localize_post_time(post_time, timezone)
    for post_time, timezone in zip(df_test_processed['post_datetime'], df_test_processed['timezone'])
]

'''MANIPULATING VARIABLES AFTER PRE-PROCESSING'''
'''ASSUMPTION: ALL OF THE DATA TYPES ARE COORDINATED AND ALIGNED'''

'''REGION'''
df_test_processed['region'] = df_test_processed['timezone'].map(timezone_to_region)
df_test_processed['region'] = df_test_processed['region'].astype('string')


'''DAY OF WEEK'''
df_test_processed['day_of_week'] = df_test_processed['post_datetime'].dt.day_name()
df_test_processed['day_of_week'] = df_test_processed['day_of_week'].astype('string')


'''AGE'''
# transform birthday to age at post using apply
df_test_processed['age'] = df_test_processed.apply(lambda row: calculate_age(row['birthday'], row['localized_post_datetime']), axis=1)
df_test_processed['age'] = pd.to_numeric(df_test_processed['age'])

'''LOCALIZED SLOT OF DAY'''
df_test_processed['localized_slot_of_day'] = df_test_processed['localized_post_datetime'].apply(get_slot_of_day)
df_test_processed['localized_slot_of_day'] = df_test_processed['localized_slot_of_day'].astype('string')


'''DAILY POST RETWEET RATE'''
df_test_processed['daily_retweet_rate'] = df_test_processed['posts_retweets'] / 30
df_test_processed['daily_retweet_rate'] = pd.to_numeric(df_test_processed['daily_retweet_rate'])



'''ACCOUNT SENIORITY (AGE AT POST) IN WEEKS'''
# calculating account seniority in weeks according to post date
df_test_processed['account_seniority_days'] = (df_test_processed['post_datetime'] - df_test_processed['account_creation_date']).dt.days
df_test_processed['account_seniority_days'] = pd.to_numeric(df_test_processed['account_seniority_days'])

df_test_processed['account_seniority_weeks'] = df_test_processed['account_seniority_days'] / 7
# replacing 0 weeks with 1 to avoid division by zero
df_test_processed['account_seniority_weeks'] = df_test_processed['account_seniority_weeks'].replace(0, 1)
df_test_processed['account_seniority_weeks'] = pd.to_numeric(df_test_processed['account_seniority_weeks'])


'''POSTS PER WEEK'''
# calculating posts per week
df_test_processed['posts_per_week'] = df_test_processed['previous_posts_count'] / df_test_processed['account_seniority_weeks']
df_test_processed['posts_per_week'] = pd.to_numeric(df_test_processed['posts_per_week'])



'''FOLLOWERS PER WEEK'''
df_test_processed['followers_per_week'] = df_test_processed['followers'] / df_test_processed['account_seniority_weeks']
df_test_processed['followers_per_week'] = pd.to_numeric(df_test_processed['followers_per_week'])


'''FOLLOWERS TO PREVIOUS POSTS RATE'''
df_test_processed['followers_to_previous_posts'] = df_test_processed['followers'] / df_test_processed['previous_posts_count']
df_test_processed['followers_to_previous_posts'] = pd.to_numeric(df_test_processed['followers_to_previous_posts'])


'''LIKES TO RETWEETS RATE'''
df_test_processed['likes_to_retweets_rate'] = (df_test_processed['post_likes'] / df_test_processed['posts_retweets'])
df_test_processed['likes_to_retweets_rate'] = pd.to_numeric(df_test_processed['likes_to_retweets_rate'])


'''LIKES TO FOLLOWERS RATE'''
df_test_processed['likes_to_followers_rate'] = df_test_processed['post_likes'] / df_test_processed['followers']
df_test_processed['likes_to_followers_rate'] = pd.to_numeric(df_test_processed['likes_to_followers_rate'])


'''TEXT WORD COUNT'''
df_test_processed['text_word_count'] = df_test_processed['text'].apply(lambda x: len(x.split()))
df_test_processed['text_word_count'] = pd.to_numeric(df_test_processed['text_word_count'])


'''TEXT # COUNT'''
df_test_processed['hashtag_count'] = df_test_processed['text'].apply(lambda x: len([word for word in x.split() if word.startswith('#')]))
df_test_processed['hashtag_count'] = pd.to_numeric(df_test_processed['hashtag_count'])


'''ATTACHED CONTENT TYPE IN POST'''
df_test_processed['attached_content_type'] = df_test_processed['embedded_content_url'].apply(classify_and_get_content_type)
df_test_processed['attached_content_type'] = df_test_processed['attached_content_type'].astype('string')


'''USER HAS PROFILE PICTURE'''
df_test_processed['user_has_profile_picture'] = df_test_processed['profile_picture'].apply(
    lambda x: 0 if pd.isna(x) or x == "?" else 1
)
df_test_processed['user_has_profile_picture'] = pd.to_numeric(df_test_processed['user_has_profile_picture'])



'''COMBINED FEATURE FOR ATTACHED CONTENT TYPE AND PICTURE'''
df_test_processed['attached_content_and_picture_status'] = df_test_processed.apply(
    lambda row: f"{row['attached_content_type']}_with_pic" if row['user_has_profile_picture'] == 1 else f"{row['attached_content_type']}_no_pic",
    axis=1
)
df_test_processed['attached_content_and_picture_status'] = df_test_processed['attached_content_and_picture_status'].astype('string')


'''COMBINED FEATURE FOR CHECKMARK (TYPE OF ACCOUNT) AND PROFILE PICTURE'''
df_test_processed['profile_picture_and_checkmark_status'] = df_test_processed.apply(
    lambda row: f"{row['user_has_profile_picture']}_{row['checkmark']}", axis=1)
df_test_processed['profile_picture_and_checkmark_status'] = df_test_processed['profile_picture_and_checkmark_status'].astype('string')


'''COMBINED FEATURE FOR PROFILE PICTURE AND USERNAME IN EMAIL'''

'''IS USERNAME DOMAIN IN EMAIL?'''
df_test_processed['username_in_email_soft'] = df_test_processed.apply(
    lambda row: username_in_email_soft(row['username'], row['email']),
    axis=1
)

df_test_processed['similar_domain_and_picture'] = df_test_processed.apply(
    lambda row: f"{'Yes' if row['user_has_profile_picture'] else 'No'} Picture / " +
                f"{'Yes' if row['username_in_email_soft'] else 'No'} Username in Email",
    axis=1
)




'''LOG SCALED VARIABLES'''
df_test_processed['log_scaled_post_likes'] = np.log(df_test_processed['post_likes'])

df_test_processed['log_scaled_posts_retweets'] = np.log(df_test_processed['posts_retweets'])

df_test_processed['log_scaled_age'] = np.log(df_test_processed['age'])

df_test_processed['log_scaled_followers'] = np.log(df_test_processed['followers'])

df_test_processed['log_scaled_previous_posts_count'] = np.log(df_test_processed['previous_posts_count'])

df_test_processed['log_scaled_account_seniority_weeks'] = np.log(df_test_processed['account_seniority_weeks'])

df_test_processed['log_scaled_followers_to_previous_posts'] = np.log(df_test_processed['followers'] / df_test_processed['previous_posts_count'])

df_test_processed['log_scaled_likes_to_retweets'] = np.log(df_test_processed['post_likes'] / df_test_processed['posts_retweets'])

df_test_processed['log_scaled_likes_to_followers'] = np.log(df_test_processed['post_likes'] / df_test_processed['followers'])

df_test_processed['log_scaled_word_count'] = np.log(df_test_processed['text_word_count'])

df_test_processed['log_scaled_hashtag_count'] = np.log(df_test_processed['hashtag_count'])

df_test_processed['log_scaled_posts_per_week'] = np.log(df_test_processed['posts_per_week'])

'''FEATURE REPRESENTATION'''

df_test_features = df_test_processed.copy()

df_test_features = df_test_features.loc[df_test_features['day_of_week'].notna()]

df_test_features = df_test_features.drop(columns=['text','username','email', 'post_datetime', 'timezone', 'embedded_content_url', 'profile_picture', 'account_creation_date', 'birthday','localized_post_datetime'])

categorical_columns = df_test_features.select_dtypes(exclude=['number']).columns
categorical_columns = [col for col in categorical_columns if col != 'id']
print(categorical_columns)


numerical_columns = df_test_features.select_dtypes(exclude=['object', 'string', 'bool']).columns

# for Categorical Columns (One-Hot Encoding)
df_test_features = pd.get_dummies(df_test_features, columns=categorical_columns)


# for Numerical Columns (Min-Max Scaling)
scaler = MinMaxScaler()

df_test_features[numerical_columns] = df_test_features[numerical_columns].replace([np.inf, -np.inf], np.nan)

# Iterate over each numerical column
for col in numerical_columns:
    max_value = df_test_features[col].max()  # Get the maximum value for the column
    df_test_features[col] = df_test_features[col].fillna(max_value)  # Fill missing values with the max value


df_test_features[numerical_columns] = scaler.fit_transform(df_test_features[numerical_columns])

print(df_test_features.columns)

df_test_with_id = df_test_features[['id','username_in_email_soft_True',
                                         'similar_domain_and_picture_Yes Picture / Yes Username in Email',
                                         'log_scaled_posts_per_week', 'hashtag_count',
                                         'log_scaled_likes_to_followers',
                                         'similar_domain_and_picture_Yes Picture / No Username in Email',
                                         'log_scaled_followers_to_previous_posts',
                                         'log_scaled_likes_to_retweets', 'log_scaled_previous_posts_count',
                                         'log_scaled_followers', 'month_of_post_December',
                                         'month_of_post_November', 'month_of_post_January',
                                         'month_of_post_February', 'checkmark_grey', 'checkmark_gold',
                                         'profile_picture_and_checkmark_status_1_gold',
                                         'profile_picture_and_checkmark_status_1_grey', 'checkmark_none',
                                         'month_of_post_June',
                                         'localized_slot_of_day_Morning']]


df_test_with_id['profile_picture_and_checkmark_status_0_none'] = 0
df_test_with_id['profile_picture_and_checkmark_status_0_none'] = df_test_with_id['profile_picture_and_checkmark_status_0_none'].astype(bool)

improved_model = joblib.load('improved_best_model.pkl')

expected_columns = improved_model.feature_names_in_

# saving the id with the matching prediction with original order of id
ids_for_pred = df_test_with_id['id']

X_test = df_test_with_id.drop(columns=['id'])

X_test = X_test[expected_columns]

# prediction
y_pred = improved_model.predict(X_test)

submission = pd.DataFrame({
      'id': ids_for_pred,
    'sentiment': np.where(y_pred == 1, 'positive', 'negative')


})

# מוודאים שהסדר לפי ה-id המקורי
submission = submission.set_index('id').loc[ids_for_pred].reset_index()

# שמירת קובץ ההגשה
submission.to_csv('group5.csv', index=False)

files.download('group5.csv')

